# QLoRA Fine-Tuning: Code Review Model

Run this notebook in **Google Colab** with a **T4 GPU** runtime
(Runtime > Change runtime type > T4 GPU, free tier).

This fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with QLoRA (4-bit) to
specialize it for code review. The 1.5B size is chosen so this comfortably
fits and trains quickly on a free T4 — bump to the 7B variant if you have
Colab Pro or an A100.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

## 1. Upload / generate the dataset

Upload `review_dataset.jsonl` (produced by `finetuning/prepare_dataset.py`)
to this Colab session, or run the cell below to regenerate the placeholder
seed dataset directly in Colab.

In [ ]:
import json, random
from pathlib import Path

SYSTEM_PROMPT = (
    "You are an expert code reviewer. Given a code diff, identify real "
    "issues (bugs, security problems, style violations, missing tests). "
    "For each issue, output: severity (low/medium/high), a one-line "
    "description, and the line(s) affected. If there are no issues, say so."
)
PROMPT_TEMPLATE = "### Instruction:\n{system_prompt}\n\n### Diff:\n{diff}\n\n### Review:\n"

seeds = [
    {
        "diff": "+ def get_user(id):\n+     query = f\"SELECT * FROM users WHERE id = {id}\"\n+     return db.execute(query)\n",
        "review": "- severity: high\n  description: SQL injection vulnerability.\n  line: 2\n  suggested_fix: use parameterized queries.\n",
    },
    {
        "diff": "+ def average(nums):\n+     return sum(nums) / len(nums)\n",
        "review": "- severity: medium\n  description: division by zero if nums is empty.\n  line: 2\n  suggested_fix: add a guard clause for empty input.\n",
    },
    {
        "diff": "+ for item in items:\n+     results.append(process(item))\n+     db.commit()\n",
        "review": "- severity: medium\n  description: committing inside a loop is inefficient.\n  line: 3\n  suggested_fix: batch commits after the loop with try/except rollback.\n",
    },
    {
        "diff": "+ def divide(a, b):\n+     return a / b\n",
        "review": "- severity: none\n  description: no issues found; consider a test for b == 0.\n",
    },
]

examples = []
for s in seeds * 20:
    text = PROMPT_TEMPLATE.format(system_prompt=SYSTEM_PROMPT, diff=s["diff"]) + s["review"]
    examples.append({"text": text})
random.shuffle(examples)

Path("review_dataset.jsonl").write_text("\n".join(json.dumps(e) for e in examples))
print(f"Wrote {len(examples)} examples")
print("\nIMPORTANT: replace this placeholder data with real curated examples ")
print("before trusting your fine-tuned vs. base-model eval comparison.")

## 2. Load base model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

## 3. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Load dataset and tokenize

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="review_dataset.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)

def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

## 5. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qlora-code-review",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=collator,
)

trainer.train()

## 6. Save the LoRA adapter

Download the `qlora-code-review-adapter` folder from Colab (or push it to
the Hugging Face Hub) and point `agent/tools/review_model.py` at it.

In [ ]:
model.save_pretrained("qlora-code-review-adapter")
tokenizer.save_pretrained("qlora-code-review-adapter")
print("Saved adapter to ./qlora-code-review-adapter")

## 7. Quick eval: fine-tuned vs. base model

This is the comparison you'll want to show in an interview — even a small
qualitative table of "base model missed this, fine-tuned model caught it"
is compelling. Run a few held-out diffs through both and compare.

In [ ]:
test_diff = (
    "+ def get_config(key):\n"
    "+     return os.environ[key]\n"
)
prompt = PROMPT_TEMPLATE.format(system_prompt=SYSTEM_PROMPT, diff=test_diff)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))